# Laboratorio 1 — Ejercicio 5
## Punto de equilibrio en la eficiencia de algoritmos

Los tiempos de ejecución de dos algoritmos, en función del tamaño del conjunto de datos $n>0$, están modelados por

$$
T_A(n)=3n\ln(n), \qquad T_B(n)=0.5n^{1.5}+10.
$$

Se busca el punto a partir del cual el tiempo del Algoritmo B supera al del Algoritmo A, aplicando el método de Bisección y documentando la evolución de los intervalos.

## Resumen del ejercicio

El punto de equilibrio se obtiene resolviendo $T_B(n)-T_A(n)=0$. Sin embargo, el intervalo $[100,1000]$ indicado en el enunciado no contiene un cambio de signo: en ambos extremos el Algoritmo B tarda menos que el A. Por ello, Bisección no puede aplicarse válidamente en ese intervalo.

La inconsistencia se documenta y luego se amplía la búsqueda al intervalo $[2000,3000]$, donde sí existe un cambio de signo. Con una tolerancia de $10^{-6}$ para el ancho del intervalo, el cruce continuo se aproxima como $n\approx2107.815991$. Como el tamaño de datos debe ser entero, el primer valor para el cual $T_B>T_A$ es $n=2108$.

## 1. Formulación del problema

Se define la función diferencia

$$
g(n)=T_B(n)-T_A(n)
    =0.5n^{1.5}+10-3n\ln(n).
$$

Una raíz de $g$ representa un punto en el que ambos tiempos son iguales. Si $g(n)<0$, entonces $T_B(n)<T_A(n)$; si $g(n)>0$, el tiempo de B ya supera al de A.

In [1]:
import math

def tiempo_a(n):
    return 3 * n * math.log(n)

def tiempo_b(n):
    return 0.5 * n**1.5 + 10

def g(n):
    return tiempo_b(n) - tiempo_a(n)

### Explicación de esta parte

Las funciones `tiempo_a` y `tiempo_b` traducen directamente los dos modelos del enunciado. La función `g` centraliza la ecuación de búsqueda de raíz y fija la interpretación del signo. Se utiliza únicamente el módulo estándar `math`, necesario para evaluar el logaritmo natural.

## 2. Revisión del intervalo indicado

Para que Bisección pueda garantizar convergencia, la función debe ser continua y sus valores en los extremos deben tener signos opuestos. Se evalúa primero el intervalo $[100,1000]$ exigido por el documento.

In [2]:
for n in (100, 1000):
    print(
        f"n = {n:4d}  "
        f"T_A(n) = {tiempo_a(n):12.6f}  "
        f"T_B(n) = {tiempo_b(n):12.6f}  "
        f"g(n) = {g(n):12.6f}"
    )

n =  100  T_A(n) =  1381.551056  T_B(n) =   510.000000  g(n) =  -871.551056
n = 1000  T_A(n) = 20723.265837  T_B(n) = 15821.388301  g(n) = -4901.877536


### Interpretación del intervalo original

Se obtiene $g(100)<0$ y $g(1000)<0$. Los dos extremos tienen el mismo signo, de modo que $[100,1000]$ **no encierra el punto de equilibrio** y ejecutar Bisección allí violaría su precondición fundamental.

Esto no significa que los modelos nunca se crucen. El término $n^{1.5}$ de $T_B$ crece finalmente más rápido que $n\ln(n)$, por lo que se debe ampliar la búsqueda hacia valores mayores de $n$.

## 3. Localización de un intervalo válido

Se prueban los extremos $2000$ y $3000$. La función $g$ es continua para $n>0$, por lo que un cambio de signo en este nuevo intervalo garantiza al menos una raíz.

In [3]:
for n in (2000, 3000):
    print(f"g({n}) = {g(n): .9f}")

g(2000) = -874.055207257
g(3000) =  10111.075516923


### Interpretación del nuevo intervalo

Ahora $g(2000)<0$ y $g(3000)>0$, así que el Teorema del Valor Intermedio garantiza un cruce en $[2000,3000]$. Además,

$$
g'(n)=0.75\sqrt{n}-3(\ln(n)+1)
$$

es positiva y creciente a partir de esta región. Por tanto, el cruce localizado es único para $n\geq2000$ y, después de él, $T_B$ permanece por encima de $T_A$.

## 4. Método de Bisección

En cada iteración se evalúa el punto medio y se conserva la mitad que mantiene el cambio de signo. El historial almacena el nuevo intervalo $[a_k,b_k]$, su punto medio, el valor de $g$ en esa aproximación y el error solicitado $|b_k-a_k|$.

Como el enunciado no proporciona una tolerancia para este ejercicio, se adopta $\varepsilon=10^{-6}$. También se fija un máximo de 100 iteraciones como salvaguarda.

In [4]:
def biseccion(funcion, a, b, tolerancia=1e-6, max_iteraciones=100):
    fa = funcion(a)
    fb = funcion(b)

    if fa == 0:
        return a, []
    if fb == 0:
        return b, []
    if fa * fb > 0:
        raise ValueError("El intervalo no presenta un cambio de signo.")

    historial = []

    for iteracion in range(1, max_iteraciones + 1):
        punto_evaluado = (a + b) / 2
        fp = funcion(punto_evaluado)

        if fa * fp <= 0:
            b = punto_evaluado
            fb = fp
        else:
            a = punto_evaluado
            fa = fp

        aproximacion = (a + b) / 2
        error = abs(b - a)
        historial.append(
            (iteracion, a, b, aproximacion, funcion(aproximacion), error)
        )

        if error < tolerancia:
            return aproximacion, historial

    raise RuntimeError("Bisección no convergió en el máximo de iteraciones.")

### Explicación del algoritmo implementado

Antes de iterar, `biseccion` comprueba si algún extremo es una raíz y rechaza intervalos sin cambio de signo. Después de cada partición registra el intervalo que todavía encierra la raíz. Así, la columna `error` coincide exactamente con |bₖ−aₖ|, solicitada en el enunciado, y se reduce a la mitad en cada paso.

La aproximación devuelta es el punto medio del último intervalo, cuya distancia a la raíz es, en realidad, como máximo la mitad del ancho mostrado.

## 5. Cálculo del punto de equilibrio

Se ejecuta Bisección en el intervalo corregido y se muestra la evolución completa de los intervalos.

In [5]:
tolerancia = 1e-6
raiz, historial = biseccion(g, 2000.0, 3000.0, tolerancia)

print(
    f"{'k':>2} {'a_k':>16} {'b_k':>16} "
    f"{'c_k':>16} {'g(c_k)':>14} {'|b_k-a_k|':>14}"
)
for k, a, b, c, gc, error in historial:
    print(
        f"{k:2d} {a:16.9f} {b:16.9f} "
        f"{c:16.9f} {gc:14.6e} {error:14.6e}"
    )

print()
print(f"Punto de equilibrio continuo: n = {raiz:.9f}")
print(f"Iteraciones necesarias: {len(historial)}")
print(f"Ancho final del intervalo: {historial[-1][5]:.6e}")

 k              a_k              b_k              c_k         g(c_k)      |b_k-a_k|
 1   2000.000000000   2500.000000000   2250.000000000   1.272308e+03   5.000000e+02
 2   2000.000000000   2250.000000000   2125.000000000   1.465948e+02   2.500000e+02
 3   2000.000000000   2125.000000000   2062.500000000  -3.770172e+02   1.250000e+02
 4   2062.500000000   2125.000000000   2093.750000000  -1.185133e+02   6.250000e+01
 5   2093.750000000   2125.000000000   2109.375000000   1.321768e+01   3.125000e+01
 6   2093.750000000   2109.375000000   2101.562500000  -5.285386e+01   1.562500e+01
 7   2101.562500000   2109.375000000   2105.468750000  -1.986957e+01   7.812500e+00
 8   2105.468750000   2109.375000000   2107.421875000  -3.338815e+00   3.906250e+00
 9   2107.421875000   2109.375000000   2108.398437500   4.936214e+00   1.953125e+00
10   2107.421875000   2108.398437500   2107.910156250   7.978955e-01   9.765625e-01
11   2107.421875000   2107.910156250   2107.666015625  -1.270661e+00   4.882

### Interpretación de la tabla

El ancho inicial de $1000$ unidades se divide entre dos en cada iteración. Después de 30 iteraciones queda por debajo de $10^{-6}$, cumpliendo el criterio elegido. Los extremos de todas las filas mantienen signos opuestos y, por tanto, continúan encerrando el punto de equilibrio.

La aproximación continua es $n\approx2107.815991$. Este valor puede ser fraccionario en el modelo matemático, pero un conjunto de datos contiene una cantidad entera de elementos; por eso todavía se debe determinar el primer entero que queda después del cruce.

## 6. Umbral entero

Se comparan los dos enteros consecutivos alrededor de la raíz. Esta verificación determina cuál es el primer tamaño de datos aplicable para el que el tiempo de B supera al de A.

In [6]:
for n in (2107, 2108):
    diferencia = g(n)
    relacion = "T_B > T_A" if diferencia > 0 else "T_B < T_A"
    print(
        f"n = {n}: g(n) = {diferencia: .9f}  "
        f"-> {relacion}"
    )

n = 2107: g(n) = -6.911638191  -> T_B < T_A
n = 2108: g(n) =  1.559225480  -> T_B > T_A


## Resultado

El intervalo $[100,1000]$ indicado en el enunciado no es válido para Bisección porque $g$ es negativa en ambos extremos. Al ampliar justificadamente la búsqueda a $[2000,3000]$, el método converge en 30 iteraciones al punto de equilibrio continuo

$$
\boxed{n\approx2107.815991}.
$$

La comprobación de los enteros vecinos muestra que $g(2107)<0$ y $g(2108)>0$. Por tanto, el primer tamaño entero a partir del cual el tiempo del Algoritmo B supera al del Algoritmo A es

$$
\boxed{n=2108}.
$$

### Conclusión

La validación del intervalo inicial es una parte indispensable del método: Bisección no debe ejecutarse solamente porque el enunciado proporcione dos extremos. Después de corregir esa inconsistencia, la reducción sistemática del intervalo y la verificación de los enteros vecinos permiten distinguir claramente entre el cruce continuo del modelo y el umbral entero con significado práctico.